## Create the engine

In [12]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

db_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password="password",
    host="localhost",
    port=5555,
    database="similarity_search_service_db",
)

engine = create_engine(db_url)

In [13]:
from typing import List
from pgvector.sqlalchemy import Vector
from sqlalchemy import Integer, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column


class Base(DeclarativeBase):
    __abstract__ = True


class Images(Base):
    __tablename__ = "images"
    VECTOR_LENGTH = 512

    # primary key
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    # image path - we will use it to store the path to the image file, after similarity search we can use it to retrieve the image and display it
    image_path: Mapped[str] = mapped_column(String(256))
    # image embedding - we will store the image embedding in this column, the image embedding is a list of 512 floats this is the output of the sentence transformer model
    image_embedding: Mapped[List[float]] = mapped_column(Vector(VECTOR_LENGTH))

In [14]:
Base.metadata.create_all(engine)

## Images similarity

In [ ]:
from sqlalchemy.orm import Session
from sqlalchemy import Engine, select
import numpy as np


# reusable function to insert data into the table
def insert_image(engine: Engine, image_path: str, image_embedding: list[float]):
    with Session(engine) as session:
        # create the image object
        image = Images(image_path=image_path, image_embedding=image_embedding)
        # add the image object to the session
        session.add(image)
        # commit the transaction
        session.commit()


# insert some data into the table
N = 100
for i in range(N):
    image_path = f"image_{i}.jpg"
    image_embedding = np.random.rand(512).tolist()
    insert_image(engine, image_path, image_embedding)

# select first image from the table
with Session(engine) as session:
    image = session.query(Images).first()


# Note: pgvector exposes *distances*, not similarities (lower = more similar).
# To rank, sort ascending by `cosine_distance`. To filter by a similarity threshold,
# use `1 - cosine_distance > threshold`.


# order the K images by ascending cosine distance to the first image (smallest distance = most similar)
def find_k_images(engine: Engine, k: int, original_image: Images) -> list[Images]:
    with Session(engine) as session:
        # execution_options={"prebuffer_rows": True} is used to prebuffer the rows, this is useful when we want to fetch the rows in chunks and return them after session is closed
        result = (
            session.execute(
                select(Images)
                .order_by(
                    Images.image_embedding.cosine_distance(
                        original_image.image_embedding
                    )
                )
                .limit(k),
                execution_options={"prebuffer_rows": True},
            )
            .scalars()
            .all()
        )
        return result


# find the 10 most similar images to the first image
k = 10
similar_images = find_k_images(engine, k, image)

In [14]:
# find images with similarity to the original above the threshold (similarity = 1 - cosine_distance)
def find_images_with_similarity_score_greater_than(
    engine: Engine, similarity_score: float, original_image: Images
) -> list[Images]:
    with Session(engine) as session:
        result = (
            session.execute(
                select(Images).filter(
                    (
                        1
                        - Images.image_embedding.cosine_distance(
                            original_image.image_embedding
                        )
                    )
                    > similarity_score
                ),
                execution_options={"prebuffer_rows": True},
            )
            .scalars()
            .all()
        )
        return result

## Games

In [7]:
from datasets import load_dataset

dataset = load_dataset("FronkonGames/steam-games-dataset")

# get columns names and types
columns = dataset["train"].features
print(columns)

columns_to_keep = [
    "name",
    "windows",
    "linux",
    "mac",
    "detailed_description",
    "supported_languages",
    "price",
]

N = 40000  # you can adjust this number
dataset = dataset["train"].select_columns(columns_to_keep).select(range(N))

{'appID': Value('string'), 'name': Value('string'), 'release_date': Value('string'), 'estimated_owners': Value('string'), 'peak_ccu': Value('int64'), 'required_age': Value('int64'), 'price': Value('float64'), 'dlc_count': Value('int64'), 'detailed_description': Value('string'), 'short_description': Value('string'), 'supported_languages': List(Value('string')), 'full_audio_languages': List(Value('string')), 'reviews': Value('string'), 'header_image': Value('string'), 'website': Value('string'), 'support_url': Value('string'), 'support_email': Value('string'), 'windows': Value('bool'), 'mac': Value('bool'), 'linux': Value('bool'), 'metacritic_score': Value('int64'), 'metacritic_url': Value('string'), 'user_score': Value('int64'), 'positive': Value('int64'), 'negative': Value('int64'), 'score_rank': Value('string'), 'achievements': Value('int64'), 'recommendations': Value('int64'), 'notes': Value('string'), 'average_playtime_forever': Value('int64'), 'average_playtime_2weeks': Value('int6

In [8]:
from sqlalchemy import Integer, Float, Boolean, Text


class Games(Base):
    __tablename__ = "games"
    __table_args__ = {"extend_existing": True}

    # the vector size produced by the model taken from documentation https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v2
    VECTOR_LENGTH = 512

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[str] = mapped_column(String(256))
    # `Text` is an unbounded string in Postgres — game descriptions can run a few KB
    description: Mapped[str] = mapped_column(Text)
    windows: Mapped[bool] = mapped_column(Boolean)
    linux: Mapped[bool] = mapped_column(Boolean)
    mac: Mapped[bool] = mapped_column(Boolean)
    price: Mapped[float] = mapped_column(Float)
    game_description_embedding: Mapped[float] = mapped_column(Vector(VECTOR_LENGTH))


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

In [9]:
from sentence_transformers import SentenceTransformer


checkpoint = "distiluse-base-multilingual-cased-v2"
model = SentenceTransformer(checkpoint)


def generate_embeddings(text: str) -> list[float]:
    return model.encode(text)

model.safetensors:   0%|          | 0.00/539M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

In [15]:
from tqdm import tqdm


def insert_games(engine, dataset):
    with tqdm(total=len(dataset)) as pbar:
        for i, game in enumerate(dataset):
            game_description = game["detailed_description"] or ""
            game_embedding = generate_embeddings(game_description)
            # keep rows with the required fields; windows/linux/mac are platform flags (True/False),
            # not missing-data signals — don't gate the insert on them or you discard most of the dataset
            if game["name"] and game_description and game["price"] is not None:
                game = Games(
                    name=game["name"],
                    description=game_description,
                    windows=game["windows"],
                    linux=game["linux"],
                    mac=game["mac"],
                    price=game["price"],
                    game_description_embedding=game_embedding,
                )
                with Session(engine) as session:
                    session.add(game)
                    session.commit()
            pbar.update(1)

In [18]:
insert_games(engine, dataset)

100%|██████████| 40000/40000 [16:04<00:00, 41.46it/s]


In [19]:
from typing import Optional


def find_game(
    engine: Engine,
    game_description: str,
    windows: Optional[bool] = None,
    linux: Optional[bool] = None,
    mac: Optional[bool] = None,
    price: Optional[int] = None,
):
    with Session(engine) as session:
        game_embedding = generate_embeddings(game_description)

        query = select(Games).order_by(
            Games.game_description_embedding.cosine_distance(game_embedding)
        )

        if price is not None:
            query = query.filter(Games.price <= price)
        if windows is not None:
            query = query.filter(Games.windows.is_(windows))
        if linux is not None:
            query = query.filter(Games.linux.is_(linux))
        if mac is not None:
            query = query.filter(Games.mac.is_(mac))

        result = session.execute(query, execution_options={"prebuffer_rows": True})
        game = result.scalars().first()

        return game

In [20]:
game = find_game(engine, "This is a game about a hero who saves the world", price=10)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

game = find_game(engine, game_description="Home decorating", price=20)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

Game: Hero's everyday life
Description: In Hero's everyday life, you need to play the role of a hero, completing a variety of tasks from give and bring, to kill monsters and bosses. Help all the different game characters with their mundane tasks and become famous throughout the fantasy kingdom as a great hero! Or just fool around and have as much fun as you can! The world is filled with various secrets, there are also enemies who are ready to break you no matter what, so arm yourself with a sword and hit the road. Who is a hero? Is this the one who gets up in the morning for an adventure? Or the one who fights for the good and prosperity? In general, you will need to decide which of you will be the hero and for what you will fight! The kingdom of fantasy is filled with monsters, NPS cannot fulfill their mission and therefore they called you a hero, and whether you are a great hero or not, it's up to you to decide! Complete a mission, fight monsters, kill bosses, or just insult the king

In [21]:
game = find_game(engine, game_description="Home decorating", mac=True, price=5)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

Game: My Universe - Interior Designer
Description: Paint colours, furnishings, style, interior design, you choose each and every detail. Talk to your clients to find out more about the hopes, dreams and preferences they have for their homes. You're going to have to renovate their houses and complete the mission objectives to satisfy their needs. The happier they are with your work, the more your notoriety and level will increase! Gradually, you'll unlock new items, styles, decorative possibilities and bigger projects. During your adventure, you can play in 2 different modes: renovation mode where you can change the structure of the house by knocking down or putting up walls, taking up wooden or tiled flooring, painting, changing the wall coverings, fixing the electrics or adding new water inlets. You can also switch to decoration mode where you can choose and position the furnishings, plants, rugs, wallpaper, curtains, etc. It's up to you to add real character to the house. And if you 

## RAG

In [24]:
from pymilvus import MilvusClient

host = "localhost"
port = "19530"

milvus_client = MilvusClient(host=host, port=port)

In [26]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(
    name="id", dtype=DataType.INT64, is_primary=True, description="Primary id"
)
text = FieldSchema(
    name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text"
)
embedding_text = FieldSchema(
    "embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=VECTOR_LENGTH,
    description="Embedded text",
)

fields = [id_field, text, embedding_text]

schema = CollectionSchema(
    fields=fields,
    auto_id=True,
    enable_dynamic_field=True,
    description="RAG Texts collection",
)

In [27]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(collection_name=COLLECTION_NAME, schema=schema)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64},  # lower values for speed
)

milvus_client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': 'RAG Texts collection', 'fields': [{'field_id': 100, 'name': 'id', 'description': 'Primary id', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': 'Page text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 102, 'name': 'embedding', 'description': 'Embedded text', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 470031941485463003, 'consistency_level': 2, 'properties': {'timezone': 'UTC'}, 'num_partitions': 1, 'enable_dynamic_field': True, 'created_timestamp': 470031966826397713, 'update_timestamp': 470031966826397713}


In [28]:
# define data source and destination
## the document origin destination from which document will be downloaded
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [29]:
# download data
import os
import requests


def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)


download_pdf_data(pdf_url, file_name)

In [30]:
import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)


extract_pdf_text(file_name, file_json)

In [31]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer


def generate_page_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append(
            {"page_num": page_num, "embedding": embeddings[page_num].tolist()}
        )

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)


model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_page_embeddings(file_json, embeddings_json, model)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with (
        open(os.path.join(data_dir, file_json), "r") as t_f,
        open(os.path.join(data_dir, embeddings_json), "r") as e_f,
    ):
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data = list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))

        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text": text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [33]:
def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings",
        data=[embedded_query],
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"],
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

## Real RAG

In [ ]:
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_KEY)

MODEL = "gemini-3.6-flash"


def generate_response(prompt: str):
    try:
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return "".join(
            part.text
            for part in response.candidates[0].content.parts
            if getattr(part, "text", None)
        )
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [108]:
generate_response("hi")

'Hello! How can I help you today?'

In [109]:
def build_prompt(context: str, query: str, style: str) -> str:
    return f"""You are a helpful assistant. Answer the question using only the context below.
If the context does not contain the answer, say that you cannot find it in the document. Use selected style: {style}.

Context:
{context}

Question: {query}
Answer:"""


def rag(model, query: str):
    query_result = search(model, query)
    hits = query_result[0] if query_result else []
    context_parts = [hit.get("entity", {}).get("text", "") for hit in hits]
    context = "\n\n".join(text for text in context_parts if text)

    if not context:
        return "Nie znaleziono pasującego fragmentu dokumentu."

    prompt = build_prompt(context, query)
    return generate_response(prompt)

## RAG prompt experiments

In [110]:
def run_rag_experiment(model, title: str, query: str, style: str = "", top_k: int = 3):
    query_embedding = model.encode(query).tolist()
    hits_by_query = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=top_k,
        search_params={"metric_type": "L2"},
        output_fields=["text"],
    )
    hits = hits_by_query[0] if hits_by_query else []
    passages = [hit.get("entity", {}).get("text", "") for hit in hits]
    context = "\n\n".join(text for text in passages if text)

    prompt = (
        build_prompt(context, query, style) if style else build_prompt(context, query)
    )
    answer = generate_response(prompt)
    return {
        "title": title,
        "query": query,
        "style": style,
        "context": context,
        "answer": answer,
    }

### Run 6 prompt variations

The tests cover style, language, format, persona, a word limit, and a hallucination probe. The first test is the baseline for comparison.

In [113]:
rag_experiment_results = []

res = run_rag_experiment(
    model,
    title="1. Baseline",
    query="Jakie korzyści może przynieść sztuczna inteligencja według przewodnika?",
    style="Answer in Polish. Keep every factual claim supported by the context; do not add pirate-themed facts.",
)
if res is not None:
    rag_experiment_results.append(res)

print(rag_experiment_results[-1]["answer"])

Według przedstawionego tekstu sztuczna inteligencja może przynieść następujące korzyści:

* **Wzrost produktywności i PKB:** Masowy przewidywany wzrost produktywności ludzi (zarówno pracowników, jak i w zakresie efektywności pozazawodowej), wzrost wartości globalnej ekonomii o 2,6 do 4,4 biliona dolarów rocznie oraz 7% wzrostu światowego PKB.
* **Wsparcie edukacji:** Personalizację i dostosowanie ścieżek edukacyjnych oraz programów nauczania, pisanie pomocy naukowych, możliwość ćwiczenia przed egzaminami, naukę nowych rzeczy (poza edukacją formalną) oraz współtworzenie programów upskillingu i reskillingu.
* **Automatyzację procesów biznesowych i finansowych:** Automatyczną ocenę zdolności kredytowej, profilowanie najlepszych klientów, planowanie kampanii reklamowych, a także zautomatyzowane zarządzanie, monitorowanie, raportowanie i rozwiązywanie problemów w procesach technologicznych.
* **Wykrywanie oszustw i analizę danych:** Prognozowanie oraz wyłapywanie podejrzanych transakcji fin

In [114]:
res = run_rag_experiment(
    model,
    title="2. Style: pirate voice, but Polish and grounded",
    query="Jakie korzyści może przynieść sztuczna inteligencja według przewodnika?",
    style="Answer in Polish using a playful pirate voice. Keep every factual claim supported by the context; do not add pirate-themed facts.",
)
if res is not None:
    rag_experiment_results.append(res)

print(rag_experiment_results[-1]["answer"])

Ahoj, majtku! Arrr! Z pism na pokładzie wynika, że sztuczna inteligencja może przynieść następujące korzyści:

* **Masowy wzrost produktywności:** Przewiduje się wzrost produktywności ludzi – zarówno w pracy, jak i efektywności pozazawodowej.
* **Wzrost wartości globalnej ekonomii:** Wedle estymacji McKinsey – wzrost na poziomie od 2,6 do 4,4 biliona dolarów rocznie, a wedle Goldman Sachs – 7% wzrostu światowego PKB.
* **Wsparcie w edukacji:** 
  * Personalizacja i dostosowanie ścieżki edukacji oraz programu nauczania do każdego człowieka.
  * Pisanie pomocy naukowych i ćwiczenie przygotowań do egzaminów.
  * Umożliwienie nauki nowych rzeczy (niekoniecznie związanych z edukacją formalną).
  * Współtworzenie programów podnoszenia i zmiany kwalifikacji (*upskillingu* i *reskillingu*).
* **Automatyzacja i usprawnienia w ekonomii oraz technologii:**
  * Automatyczna ocena zdolności kredytowej, profilowanie najlepszych klientów i planowanie kampanii reklamowych.
  * Zautomatyzowane zarządza

In [115]:
res = run_rag_experiment(
    model,
    title="3. Format: exactly three bullet points",
    query="Jakie zastosowania sztucznej inteligencji opisuje przewodnik?",
    style="Odpowiedz po polsku dokładnie w trzech punktach. Każdy punkt musi wynikać z kontekstu.",
)
if res is not None:
    rag_experiment_results.append(res)

print(rag_experiment_results[-1]["answer"])

Oto zastosowania sztucznej inteligencji opisane w tekście:

1. **Finanse, ekonomia i biznes:** automatyczna ocena zdolności kredytowej, profilowanie klientów, planowanie kampanii reklamowych oraz prognozowanie i wykrywanie oszustw (np. podejrzanych transakcji finansowych).
2. **Zarządzanie, monitoring i analiza wideo:** zautomatyzowane zarządzanie i raportowanie w procesach technologicznych, systemy monitoringu w czasie rzeczywistym oraz zarządzanie i prognozowanie ruchu samochodowego i pieszego.
3. **Generowanie treści, prognozowanie i interakcja:** pisanie tekstów i kodów programów, tworzenie oraz modyfikacja obrazów, wideo (deepfake) i dźwięków, prognozowanie pogody, wyników wyborów i cen akcji, a także interakcja z użytkownikami przez chatboty i asystentów wirtualnych.


In [116]:
res = run_rag_experiment(
    model,
    title="4. Persona: skeptical reviewer",
    query="Jakie zagrożenia lub ograniczenia sztucznej inteligencji opisuje przewodnik?",
    style="Wciel się w sceptycznego recenzenta. Dla każdej tezy dodaj etykietę [supported] albo [unsupported]. Oznacz [supported] tylko te tezy, które wynikają wprost z kontekstu; jeśli kontekst nie pozwala ocenić tezy, napisz to.",
)
if res is not None:
    rag_experiment_results.append(res)

print(rag_experiment_results[-1]["answer"])

Jako sceptyczny recenzent muszę zauważyć, że dostarczony tekst jedynie w sposób powierzchowny i wybiórczy traktuje temat zagrożeń oraz ograniczeń technologii. Na podstawie przedstawionego materiału można sformułować następujące wnioski:

1. Wąska (słaba) sztuczna inteligencja posiada istotne ograniczenia – jest pozbawiona zdolności ogólnego rozumienia, nie posiada świadomości ani emocji i działa wyłącznie w ramach wcześniej ustalonej struktury ograniczonej do konkretnego zadania [supported].
2. Generatywna sztuczna inteligencja stwarza bezpośrednie zagrożenia dla bezpieczeństwa i etyki, ze szczególnym uwzględnieniem ryzyka generowania fałszywych informacji oraz tworzenia deepfake'ów [supported].
3. Ogólny rozwój AI wiąże się z ryzykiem wystąpienia niepożądanych skutków oraz naruszenia wartości etycznych, co wymaga stosowania specjalnych środków minimalizacji ryzyka (w ramach tzw. bezpiecznej AI) [supported].

Dokument nie przedstawia żadnych konkretnych danych ilościowych na temat skal

In [117]:
res = run_rag_experiment(
    model,
    title="5. Constraint: max 50 words",
    query="Jak przewodnik opisuje odpowiedzialne korzystanie ze sztucznej inteligencji?",
    style="Odpowiedz po polsku w maksymalnie 50 słowach. Nie dodawaj informacji spoza kontekstu.",
)
if res is not None:
    rag_experiment_results.append(res)

print(rag_experiment_results[-1]["answer"])

Nie mogę znaleźć tej informacji w dokumencie.


In [118]:
res = run_rag_experiment(
    model,
    title="6. Hallucination probe: information absent from the document",
    query="Jaki jest ulubiony kolor autora przewodnika i jaką kawę pije podczas pracy?",
    style="Odpowiedz po polsku. Jeśli odpowiedzi nie ma w dostarczonym kontekście, powiedz wprost: 'Tego nie ma w źródłach'. Nie zgaduj.",
)
if res is not None:
    rag_experiment_results.append(res)

print(rag_experiment_results[-1]["answer"])

Tego nie ma w źródłach.


### Observation notes

Based on the outputs above. “Grounded” means the answer agrees with the retrieved passages; a refusal alone can also mean retrieval missed relevant text, so check the context before concluding the document lacks the answer.

| Experiment | Stayed grounded? | What changed or drifted? |
|---|---|---|
| Baseline | Mostly yes | Answered in Polish and summarized benefits. The figures and examples should be checked against the retrieved passages. |
| Pirate style | Mostly yes | The pirate voice survived (“Ahoj”, “Arrr”) while the substantive claims stayed close to the baseline. |
| Three bullet points | Mostly; verify examples | Produced exactly three numbered points. It introduced several specific examples (weather, elections, stock prices, chatbots); confirm each appears in the retrieved passages. |
| Skeptical reviewer | Partly; verify claims | Used the requested supported/unsupported labels, but also added evaluative statements and broad claims. Check every “supported” label against the context. |
| Max 50 words | Not enough evidence | Returned a short refusal. This may be grounded, but the query may also have retrieved irrelevant passages; inspect the retrieved context before treating it as proof the document lacks the answer. |
| Hallucination probe | Yes | Correctly said the author's favorite color and coffee are not in the sources, without inventing an answer. |
